# Bacterial Growth Analysis

## Data Quality

### Objective

The objective of this stage is to assess the quality, consistency, and
completeness of the dataset before performing exploratory and statistical
analysis.

The analysis will focus on identifying missing values, duplicate
records, duplicated time points, implausible values, and inconsistencies
within experimental conditions.


In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
df = pd.read_csv("data/raw/bacteria_growth.csv")
df.shape


## Missing Values

The dataset contains 136 missing values in the `population` variable,
representing approximately 2.29% of all observations.

The distribution of these missing values is examined across
experiments, experimental conditions, and time.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

missing_summary


In [ ]:
missing_population = df[df["population"].isna()]

missing_by_experiment = (
    missing_population.groupby("experiment_id")
    .size()
)

missing_by_hour = (
    missing_population.groupby("hour")
    .size()
    .sort_index()
)

missing_by_condition = (
    missing_population.groupby(["strain", "temperature"])
    .size()
    .sort_values(ascending=False)
)

print(f"Missing population records: {len(missing_population)}")
print(f"Experiments affected: {missing_by_experiment.size}")

print("\nMissing records by experiment:")
print(missing_by_experiment.value_counts().sort_index())

print("\nMissing records by hour:")
print(missing_by_hour)

print("\nMissing records by strain and temperature:")
print(missing_by_condition)




### Missing Values Findings

A total of 136 `population` values are missing, representing 2.29% of
the dataset.

The missing values affect 84 of the 120 experiments and occur across
different experimental conditions and time points.

The distribution of missing values will be further evaluated to
determine whether the missingness appears systematic or random before
selecting an appropriate treatment strategy.


## Duplicate Records

The data understanding stage identified exact duplicate records.
This section evaluates their extent and distribution before deciding
whether they should be removed from the analysis dataset.


In [ ]:
duplicate_mask = df.duplicated(keep=False)

duplicate_count = df.duplicated().sum()
duplicate_rows = df[duplicate_mask]

duplicate_experiments = (
    duplicate_rows
    .groupby("experiment_id")
    .size()
)

print(f"Exact duplicate records: {duplicate_count}")
print(f"Rows involved in duplication: {len(duplicate_rows)}")
print(f"Experiments affected: {duplicate_experiments.size}")


### Duplicate Time Points

Each experiment is expected to contain at most one observation for each
hour.

Duplicate time points are therefore evaluated to determine whether they
represent repeated measurements or duplicated records.


In [ ]:
duplicate_timepoints = (
    df.groupby(["experiment_id", "hour"])
    .size()
    .reset_index(name="count")
)

duplicate_timepoints = duplicate_timepoints[
    duplicate_timepoints["count"] > 1
]

duplicate_timepoints.head(10)


In [ ]:
print(f"Duplicated experiment-hour combinations: {len(duplicate_timepoints)}")
print(
    f"Additional records caused by repeated time points: "
    f"{(duplicate_timepoints['count'] - 1).sum()}"
)


In [ ]:
duplicate_timepoints_check = (
    df.groupby(["experiment_id", "hour"])
    .agg(
        records=("population", "size"),
        population_unique=("population", "nunique"),
        growth_rate_unique=("growth_rate", "nunique"),
        capacity_unique=("capacity", "nunique")
    )
)

duplicate_timepoints_check[
    duplicate_timepoints_check["records"] > 1
].head(10)


### Duplicate Findings

The dataset contains 59 exact duplicate records, involving 118 rows
across 44 experiments.

The duplicated records correspond to 59 repeated `experiment_id` and
`hour` combinations. Each duplicated time point contains two identical
records, indicating that these observations are redundant rather than
independent measurements.

Therefore, the duplicated records can be safely removed from the
analysis dataset.

In [ ]:
df_clean = df.drop_duplicates().copy()

print(f"Original rows: {len(df)}")
print(f"Rows after removing duplicates: {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)}")


## Value Range Checks

The numerical variables are assessed for values outside their expected
ranges and for potential inconsistencies with the structure of the
dataset.

In [ ]:
numeric_summary = df_clean[
    ["temperature", "hour", "population", "growth_rate", "capacity"]
].describe()

range_checks = pd.Series({
    "negative_population": (df_clean["population"] < 0).sum(),
    "negative_growth_rate": (df_clean["growth_rate"] < 0).sum(),
    "negative_capacity": (df_clean["capacity"] < 0).sum(),
    "hour_below_zero": (df_clean["hour"] < 0).sum(),
    "hour_above_48": (df_clean["hour"] > 48).sum(),
})

population_above_capacity = df_clean[
    df_clean["population"] > df_clean["capacity"]
]

print("Summary statistics:")
display(numeric_summary)

print("\nRange violations:")
display(range_checks)

print(
    f"\nObservations above capacity: "
    f"{len(population_above_capacity)}"
)


In [ ]:
above_capacity = population_above_capacity.copy()

above_capacity["excess_ratio"] = (
    above_capacity["population"] /
    above_capacity["capacity"]
)

display(
    above_capacity["excess_ratio"].describe()
)

display(
    above_capacity
    .groupby(["strain", "temperature"])["excess_ratio"]
    .agg(["count", "mean", "max"])
)

above_capacity.loc[
    above_capacity["excess_ratio"].idxmax()
]


In [ ]:
above_capacity[
    above_capacity["excess_ratio"] > 2
].sort_values(
    "excess_ratio",
    ascending=False
)


### Findings

A total of 492 observations have population values above the reported
carrying capacity.

Most of these observations are only slightly above capacity, with a
median population-to-capacity ratio of approximately 1.02. However,
two observations show substantially larger deviations, with population
values exceeding two times the reported capacity.

The most extreme case occurs in experiment 62, where the population is
approximately 5.04 times the carrying capacity. Since the dataset is
synthetic, these observations are not classified as invalid solely
based on this comparison. They will be retained for further analysis
and considered when interpreting the growth dynamics.


## Experimental Consistency

The consistency of experimental conditions within each `experiment_id`
is assessed to verify that strain, temperature, growth rate, and
carrying capacity remain constant throughout each experiment.


In [ ]:
experiment_consistency = df_clean.groupby("experiment_id").agg(
    strain_unique=("strain", "nunique"),
    temperature_unique=("temperature", "nunique"),
    growth_rate_unique=("growth_rate", "nunique"),
    capacity_unique=("capacity", "nunique")
)

experiment_consistency


In [ ]:
experiment_consistency[
    (experiment_consistency > 1).any(axis=1)
]


### Findings

The consistency check confirms that `strain`, `temperature`, `growth_rate`,
and `capacity` remain constant within each experiment. No inconsistent
experimental conditions were identified.



## Data Analysis Summary

The data quality assessment identified missing population values and
exact duplicate records as the main data quality issues. Basic range
checks and experimental consistency checks did not identify invalid
values or conflicting experimental conditions.

In [ ]:
quality_summary = pd.DataFrame({
    "metric": [
        "Original rows",
        "Rows after removing duplicates",
        "Exact duplicate records",
        "Missing population values",
        "Experiments",
        "Observations above capacity"
    ],
    "value": [
        len(df),
        len(df_clean),
        df.duplicated().sum(),
        df_clean["population"].isna().sum(),
        df_clean["experiment_id"].nunique(),
        (df_clean["population"] > df_clean["capacity"]).sum()
    ]
})

quality_summary

### Findings

The dataset contains 5,939 original observations. After removing 59
exact duplicate records, 5,880 observations remain.

The `population` variable contains 136 missing values, which will
require consideration before the analysis of population dynamics.

No basic range violations or inconsistent experimental conditions were
identified. Observations above the reported carrying capacity were
retained because their validity cannot be determined solely from this
comparison in the synthetic dataset.

## 6. Missing Population Treatment

The missing `population` values require treatment because population
will be the primary variable used to analyse bacterial growth over time.

The treatment should preserve the temporal structure of the experiments
without introducing unsupported values.

In [ ]:
missing_by_experiment = (
    df_clean[df_clean["population"].isna()]
    .groupby("experiment_id")
    .agg(
        missing_count=("population", "size"),
        first_missing_hour=("hour", "min"),
        last_missing_hour=("hour", "max")
    )
)

missing_by_experiment.head(10)

In [ ]:
missing_by_experiment["missing_count"].value_counts().sort_index()

In [ ]:
missing_sequences = (
    df_clean[df_clean["population"].isna()]
    .groupby("experiment_id")["hour"]
    .apply(lambda x: sorted(x.tolist()))
)

missing_sequences.head(20)

### Missing Value Decision

The 136 missing `population` values are distributed across 84
experiments. Most affected experiments contain one or two missing
observations, although a small number contain consecutive missing time
points.

Because `population` is the primary variable used to analyse growth
dynamics, imputing missing values could introduce artificial observations
into the analysis. Therefore, the missing values will be retained as
`NaN` and handled according to the requirements of each subsequent
analysis.

In [ ]:
print(f"Rows: {len(df_clean)}")
print(f"Missing population: {df_clean['population'].isna().sum()}")
print(f"Exact duplicates: {df_clean.duplicated().sum()}")

In [41]:
df_clean.to_csv(
    "data/processed/bacteria_growth_clean.csv",
    index=False
)

## Data Quality Summary

The data quality assessment identified 59 exact duplicate records and
136 missing `population` values.

The duplicate records were removed from the analysis dataset, while the
raw dataset was preserved unchanged. Missing population values were
retained as `NaN` because imputing the primary measurement could
introduce artificial observations into the growth analysis.

Basic range and experimental consistency checks did not identify
invalid values or conflicting experimental conditions.

The resulting analysis dataset contains 5,880 observations, with 136
missing population values and no exact duplicate records.